# Lay Out the Data: from small files to liquid clustering

**Big Data / Databricks · Topic 24 · Lab 2401 · Storage: read less**

**Question:** data skipping uses the min/max of each file. How do we organise the files so that most of them can be skipped, and what does each tool cost when new data arrives?

| Part | Step | What we do | Evidence |
|---|---|---|---|
| A · mechanics (1 M rows) | 1 | Write a table as **200 small files** | `numFiles`, average file size |
| | 2 | `OPTIMIZE`: compact | files removed / added in one commit |
| | 3 | Look inside: min/max per file | compacted, but **still unordered** |
| | 4 | `OPTIMIZE ZORDER BY (id)` | narrow id ranges |
| | 5 | New data arrives + `ZORDER` again | **every** file rewritten |
| | 6 | `PARTITIONED BY (year)` | one folder per year; useless for `id` |
| | 7 | Liquid clustering: `CLUSTER BY (id)` + `OPTIMIZE` | 1 file → narrow files |
| | 8 | New data arrives + `OPTIMIZE` again | **only the new data** rewritten |
| B · proof (large tables) | 9 | Same query, unordered vs clustered, cold and warm, in the **Query Profile** | files pruned, bytes read, task time |

> **Part A** uses small *path tables* inside a volume: neither managed nor external, just a folder that Delta reads and writes. We use them **only so we can look at their files**. A managed table has exactly the same Parquet files and `_delta_log`; Unity Catalog just hides them.
> **Part B** uses Unity Catalog **managed** tables, as you would in production.

## Setup

**Compute:** Serverless.
The environment notebook creates the schema, empties the three folders of Part A, builds (or reuses) the two large tables of Part B and defines three helpers. The first run takes a few minutes; afterwards, seconds.

In [ ]:
%run "./2401-00-create-environment"

Everything this notebook touches, written out in full (the SQL cells below use these literal names):

| Name | What |
|---|---|
| `/Volumes/lab2401/layout/files/zordered` | Part A · steps 1–5 · small files → OPTIMIZE → Z-order |
| `/Volumes/lab2401/layout/files/partitioned` | Part A · step 6 · partitioned by `year` |
| `/Volumes/lab2401/layout/files/clustered` | Part A · steps 7–8 · liquid clustering by `id` |
| `lab2401.layout.orders_unordered` | Part B · large table, unordered |
| `lab2401.layout.orders_clustered` | Part B · same rows, clustered by `order_date` |

# Part A · The mechanics, one tool at a time

1 M rows, three columns: `id` (0 → 999,999), `year` (2000 + `id` % 24) and `salary`.
Every table of this part has **auto compaction and optimized writes off** and a target file size of **1 MB**: the layout you see is the layout we wrote, and the files are small enough that 1 M rows give several of them to look at.

> **`delta.targetFileSize` = 1 MB is a demo trick.** In production, 1 MB files are the small-files problem itself. Leave the size to Databricks' autotuning.

## 1 · Many small files

A table written as many tiny files, like a job that runs every few minutes or a write with too many tasks. Here, one `INSERT` spread over **200 tasks** (`REPARTITION(200)`), each writing its own file.

In [ ]:
CREATE TABLE delta.`/Volumes/lab2401/layout/files/zordered` (id BIGINT, year INT, salary INT)
USING DELTA
TBLPROPERTIES (
  'delta.autoOptimize.optimizeWrite' = 'false',   -- no shuffle before the write: every task writes its own file
  'delta.autoOptimize.autoCompact'   = 'false',   -- no automatic compaction: we run OPTIMIZE ourselves
  'delta.targetFileSize'             = '1048576'  -- 1 MB, so OPTIMIZE produces several files (demo only)
);

INSERT INTO delta.`/Volumes/lab2401/layout/files/zordered`
SELECT /*+ REPARTITION(200) */
       id,
       CAST(2000 + id % 24 AS INT)    AS year,
       CAST(rand() * 100000 AS INT)   AS salary
FROM range(1000000);

🔮 **Predict:** how many files, and how big is each one?

**Do I have a small-files problem?** Divide `sizeInBytes` by `numFiles` from `DESCRIBE DETAIL`: that is the **average file size**.
- A few **MB** or less per file → too many small files: every query opens more files than it should, and the log grows with every one of them.
- **Hundreds of MB** per file → healthy.

In [ ]:
DESCRIBE DETAIL delta.`/Volumes/lab2401/layout/files/zordered`;

In [ ]:
%python
file_stats(path_table(ZORDERED))     # the same numbers, with the average already computed

Spark does **not** launch one task per small file when it reads them: it packs several small files into one task (each file counts as at least 4 MB, `openCostInBytes`). The cost is **more opening per task, and a bigger log**, not more tasks.

## 2 · OPTIMIZE: compact

🔮 **Predict:** what will the OPTIMIZE commit contain?

In [ ]:
OPTIMIZE delta.`/Volumes/lab2401/layout/files/zordered`;

In [ ]:
-- The commit that OPTIMIZE just wrote. In operationMetrics: numRemovedFiles (out) and numAddedFiles (in)
DESCRIBE HISTORY delta.`/Volumes/lab2401/layout/files/zordered` LIMIT 1;

In [ ]:
%python
file_stats(path_table(ZORDERED))

**Many out, few in**, in one atomic commit (`dataChange = false`): same rows, different files. Readers keep their version until the commit lands. The old small files stay on disk, marked as removed in the log, until `VACUUM` deletes them.

> **OPTIMIZE fixes the size of the files.** Did it also put similar ids together?

## 3 · Look inside the files: min/max per file

🔮 **Predict:** after compaction, which files could a query `WHERE id = 123456` skip?

In [ ]:
SELECT _metadata.file_name AS file,
       count(*) AS num_rows,
       min(id)  AS id_min,
       max(id)  AS id_max
FROM delta.`/Volumes/lab2401/layout/files/zordered`
GROUP BY 1
ORDER BY id_min;

**Compacted, but still unordered.** Every file still spans (almost) **0 → 999,999**: the 200 small files were packed together as they came, not sorted. For `id = 123456` the min/max of **every** file says "maybe here", so nothing is skipped.

Compaction and ordering are two different jobs. Ordering is next.

## 4 · OPTIMIZE ZORDER BY (id): put similar ids together

🔮 **Predict:** what will the id range of each file look like now?

In [ ]:
OPTIMIZE delta.`/Volumes/lab2401/layout/files/zordered` ZORDER BY (id);

In [ ]:
%python
show_files(path_table(ZORDERED), "id")     # same query as step 3

**Narrow, non-overlapping ranges.** Now `id = 123456` falls in one file's range, and the log lets Spark skip all the others **without opening them**.

(With one column, Z-order is simply a sort. Its real point is several columns at once, where it keeps each of them reasonably close.)

## 5 · New data arrives, and Z-order again

200,000 new rows with ids anywhere between 0 and 1 M, written as **one** file.

🔮 **Predict:** what is the range of the new file? And when we run `ZORDER` again, which files will it rewrite: only the new one, or all of them?

In [ ]:
INSERT INTO delta.`/Volumes/lab2401/layout/files/zordered`
SELECT CAST(rand() * 1000000 AS BIGINT)  AS id,
       2000 + CAST(rand() * 24 AS INT)    AS year,
       CAST(rand() * 100000 AS INT)       AS salary
FROM range(0, 200000, 1, 1);             -- 1 partition → 1 file

In [ ]:
%python
show_files(path_table(ZORDERED), "id")

The new file spans **0 → 1 M** and overlaps every sorted file: for any id, Spark must open it. Skipping is degraded for the whole table until we sort again.

In [ ]:
OPTIMIZE delta.`/Volumes/lab2401/layout/files/zordered` ZORDER BY (id);

In [ ]:
%python
last_commits(path_table(ZORDERED), 4)     # the whole story: compact, Z-order, new data, Z-order again

**Every file was rewritten**, not just the new one. Z-order sorts per partition, and a partition that received new data is sorted again **entirely**. This table has no partitions, so the whole table is one partition.

On a large table that means: every load → unsorted data → `OPTIMIZE ZORDER` again → the whole table rewritten again. **Sorting works; re-sorting everything on every load does not scale.** Keep this number in mind for step 8.

## 6 · Partitioning: one folder per value

Same 1 M rows, now `PARTITIONED BY (year)`.

🔮 **Predict:** a query `WHERE year = 2010` and a query `WHERE id = 123456`. Which one does partitioning help?

In [ ]:
CREATE TABLE delta.`/Volumes/lab2401/layout/files/partitioned`
USING DELTA
PARTITIONED BY (year)
TBLPROPERTIES ('delta.autoOptimize.autoCompact' = 'false')
AS SELECT id,
          CAST(2000 + id % 24 AS INT)    AS year,
          CAST(rand() * 100000 AS INT)   AS salary
   FROM range(1000000);

One folder per year:

In [ ]:
%python
display(dbutils.fs.ls(PARTITIONED))

In [ ]:
%python
show_files(path_table(PARTITIONED), "id")

- `WHERE year = 2010` → Spark reads **one folder** and skips the other 23 by the folder name. Partitioning is perfect for that.
- `WHERE id = 123456` → every file spans (almost) the **whole** id range, because `year = 2000 + id % 24` puts ids from everywhere in every year. **Nothing can be skipped.**

Partitioning organises by **one** low-cardinality column, with folders. For any other column it does not help, and choosing a high-cardinality column (an id, a timestamp) creates thousands of folders of tiny files.

## 7 · Liquid clustering: CLUSTER BY (id)

New table, same data, `CLUSTER BY (id)` and no partitions.

> This CTAS is also **how you migrate a partitioned table**: partitioning and liquid clustering are incompatible, so `ALTER TABLE … CLUSTER BY` on a partitioned table fails. You rewrite the data into a new table with `CLUSTER BY`.

In [ ]:
CREATE TABLE delta.`/Volumes/lab2401/layout/files/clustered`
USING DELTA
CLUSTER BY (id)
TBLPROPERTIES (
  'delta.targetFileSize'           = '1048576',  -- 1 MB, so OPTIMIZE produces several files (demo only)
  'delta.autoOptimize.autoCompact' = 'false'     -- no automatic compaction: we run OPTIMIZE ourselves
)
AS SELECT * FROM delta.`/Volumes/lab2401/layout/files/partitioned`;

🔮 **Predict:** clustering is declared. Are the files already sorted by id?

In [ ]:
%python
show_files(path_table(CLUSTERED), "id")

**One single file with every id.** Two things happened:

1. **`CLUSTER BY` does not cluster small writes.** A write is clustered on the way in only if it is large: **64 MB** per transaction on managed tables and **256 MB** on other tables (one clustering column). Our CTAS is a few MB, so `CLUSTER BY` is, for now, only an intention recorded in the table metadata. **Below the threshold, `OPTIMIZE` does the clustering; above it, the write itself already clusters** (the large clustered table of Part B, over a GB in one write, is above it).
2. **The write produced one file.** The data is small and Databricks combined the output into a single file while writing.

**`CLUSTER BY` declares; `OPTIMIZE` does.**

In [ ]:
OPTIMIZE delta.`/Volumes/lab2401/layout/files/clustered`;

In [ ]:
%python
show_files(path_table(CLUSTERED), "id")
last_commits(path_table(CLUSTERED))

Several files of about 1 MB (the `targetFileSize`), each covering **its own id range**, without overlaps. Same picture as Z-order in step 4. The difference shows up when new data arrives.

> **Who runs OPTIMIZE in real life?** On Unity Catalog **managed** tables with **Predictive Optimization**, Databricks runs it for you. Otherwise (managed without it, external, or path tables like this one) **you** run or schedule it. There is no "clustering score" to look at: what you can observe is the min/max of each file, as here.

## 8 · New data arrives, and OPTIMIZE again

### 8.1 A batch with a narrow range

200,000 rows with ids between 200,000 and 400,000, as one file.

🔮 **Predict:** where will they land? What will the next `OPTIMIZE` rewrite?

In [ ]:
INSERT INTO delta.`/Volumes/lab2401/layout/files/clustered` (id, year, salary)
SELECT CAST(200000 + rand() * 200000 AS BIGINT)  AS id,
       2000 + CAST(rand() * 24 AS INT)            AS year,
       CAST(rand() * 100000 AS INT)               AS salary
FROM range(0, 200000, 1, 1);                     -- 1 partition → 1 file

In [ ]:
%python
show_files(path_table(CLUSTERED), "id")

The new file covers 200K–400K and overlaps two or three existing files. It lands **unclustered**: a few MB is far below the 256 MB threshold for clustering on write.

In [ ]:
OPTIMIZE delta.`/Volumes/lab2401/layout/files/clustered`;

In [ ]:
%python
last_commits(path_table(CLUSTERED))
show_files(path_table(CLUSTERED), "id")

**Look at `files_removed` and `files_added`.** Liquid clustering decides **file by file** whether something still needs clustering, and records that decision in the log. Its range is already narrow, so queries still skip well: only around 200K–400K do two or three files overlap.

<details><summary><b>Curious?</b> The add action of that INSERT, as the log stores it</summary>

Version 0 is the CTAS, 1 the first `OPTIMIZE`, 2 this `INSERT` (check `DESCRIBE HISTORY` if your numbers differ). The `stats` field holds the min/max that data skipping reads; clustering information, when present, also lives in the add action.

```sql
SELECT value
FROM text.`/Volumes/lab2401/layout/files/clustered/_delta_log/00000000000000000002.json`
WHERE value LIKE '{"add"%';
```
</details>

So what happens when new data is spread over the **whole** range, as in step 5?

### 8.2 Two batches spread over the whole range

Two appends of 200,000 rows each, ids anywhere between 0 and 1 M: the same kind of data that forced Z-order to rewrite everything.

🔮 **Predict:** what will `OPTIMIZE` rewrite this time: the whole table, or only the new data?

In [ ]:
INSERT INTO delta.`/Volumes/lab2401/layout/files/clustered` (id, year, salary)
SELECT CAST(rand() * 1000000 AS BIGINT)  AS id,
       2000 + CAST(rand() * 24 AS INT)    AS year,
       CAST(rand() * 100000 AS INT)       AS salary
FROM range(0, 200000, 1, 1);

INSERT INTO delta.`/Volumes/lab2401/layout/files/clustered` (id, year, salary)
SELECT CAST(rand() * 1000000 AS BIGINT)  AS id,
       2000 + CAST(rand() * 24 AS INT)    AS year,
       CAST(rand() * 100000 AS INT)       AS salary
FROM range(0, 200000, 1, 1);

In [ ]:
%python
show_files(path_table(CLUSTERED), "id")

Two new files spanning **0 → 1 M**: for any id, Spark must open them. Skipping is degraded for the whole table, exactly as in step 5.

In [ ]:
OPTIMIZE delta.`/Volumes/lab2401/layout/files/clustered`;

In [ ]:
%python
last_commits(path_table(CLUSTERED))
show_files(path_table(CLUSTERED), "id")

**Only the new data was rewritten.** The two wide files were replaced by a few narrow ones; the original files kept their names, untouched.

| Same kind of new data | Files rewritten by the next OPTIMIZE |
|---|---|
| Z-order (step 5) | **all** of them |
| Liquid clustering (step 8.2) | **only the new ones** |

**The cost of OPTIMIZE follows what changed, not the size of the table.**

Old and new files now overlap a little (two "generations"), but every file is narrow again, so any id lives in one or two files instead of all of them. To get one single clean layout again (after a big backfill, or after changing the clustering key) there is `OPTIMIZE … FULL` (Databricks Runtime 16.0+), which reclusters the whole table.

# Part B · The proof: unordered vs clustered in the Query Profile

Part A showed the files. Part B shows what the layout is worth, on two large **managed** tables built by the setup (200 M rows at `medium`):

| Table | Layout |
|---|---|
| `lab2401.layout.orders_unordered` | every file holds orders from **all** dates |
| `lab2401.layout.orders_clustered` | same rows, `CLUSTER BY (order_date)` + `OPTIMIZE`: every file holds a **narrow** date range |

The setup generates the two tables **separately** with the same deterministic function: same rows, and the setup never reads either of them, so **both start cold** in the disk cache.

And one everyday query: **orders, distinct customers and revenue per country, for one week.**
Two choices make the layout visible in the **time**, not only in the file counts:
- **A short window (one week)**: the clustered table needs one file; the unordered one still has to open all of them.
- **Four columns** (`order_date`, `country`, `customer_id`, `amount`): every file the unordered table cannot skip costs four columns of reading, not one.

## 9 · Same query, two layouts

### 9.1 Predict from the metadata

`file_stats` only reads the table metadata (`DESCRIBE DETAIL`), not the data, so it does not warm anything up.

🔮 **Predict:** for one week of March 2024, how many files of each table will be read?

In [ ]:
%python
file_stats(ORDERS_UNORDERED)
file_stats(ORDERS_CLUSTERED)

### 9.2 Two caches, only one can be switched off

| Cache | What it keeps | Can we switch it off? | Where you see it |
|---|---|---|---|
| **Result cache** | The **result** of a query already run with the same text | Yes: `SET use_cached_result = false` | If it hits, the query does not run: no Scan, no files read |
| **Disk cache** | Local copies of the **Parquet files** already read, on the executors' disks | **No**, not on serverless | Query Profile → **Bytes read from cache** |

We switch off the result cache, because we are going to run each query twice. The disk cache stays: that is why we compare **run 1 with run 1** (both cold) and **run 2 with run 2** (both warm), never a cold run with a warm one.

**Pruning does not depend on any cache:** files and bytes pruned are decided from the stats in the log before anything is read.

In [ ]:
-- Switch off the result cache for this session, so every run really executes.
-- If your compute rejects this setting, comment it out: then change the dates for each run instead.
SET use_cached_result = false;

### 9.3 Run 1: both tables cold

In [ ]:
-- A · unordered · run 1 (cold)
SELECT country,
       count(*)                    AS orders,
       count(DISTINCT customer_id) AS customers,
       round(sum(amount), 2)       AS revenue
FROM lab2401.layout.orders_unordered
WHERE order_date >= DATE'2024-03-04'
  AND order_date <  DATE'2024-03-11'
GROUP BY country
ORDER BY revenue DESC;

In [ ]:
-- B · clustered · run 1 (cold): the same query, only the table name changes
SELECT country,
       count(*)                    AS orders,
       count(DISTINCT customer_id) AS customers,
       round(sum(amount), 2)       AS revenue
FROM lab2401.layout.orders_clustered
WHERE order_date >= DATE'2024-03-04'
  AND order_date <  DATE'2024-03-11'
GROUP BY country
ORDER BY revenue DESC;

**Same result, row by row.** Only the layout differs.

### 9.4 Run 2: both tables warm

Exactly the same two queries again. Now the files each one read in run 1 are in the disk cache.

In [ ]:
-- A · unordered · run 2 (warm)
SELECT country,
       count(*)                    AS orders,
       count(DISTINCT customer_id) AS customers,
       round(sum(amount), 2)       AS revenue
FROM lab2401.layout.orders_unordered
WHERE order_date >= DATE'2024-03-04'
  AND order_date <  DATE'2024-03-11'
GROUP BY country
ORDER BY revenue DESC;

In [ ]:
-- B · clustered · run 2 (warm)
SELECT country,
       count(*)                    AS orders,
       count(DISTINCT customer_id) AS customers,
       round(sum(amount), 2)       AS revenue
FROM lab2401.layout.orders_clustered
WHERE order_date >= DATE'2024-03-04'
  AND order_date <  DATE'2024-03-11'
GROUP BY country
ORDER BY revenue DESC;

### 9.5 Open the four Query Profiles

For each of the four cells:
1. Under the cell, click **See performance** → the query → **View query profile**.
2. In the **Details** panel, read the **IO** and **Files & partitions** sections, and the two clocks: **Total wall-clock duration** and **Aggregated task time**.
3. Open the **Scan Table** node: the same file and byte counts, at operator level.

| Evidence | A · run 1 | B · run 1 | A · run 2 | B · run 2 |
|---|---|---|---|---|
| Files read | | | | |
| Files pruned | | | | |
| Bytes read | | | | |
| Bytes read from cache | | | | |
| Rows read | | | | |
| Aggregated task time | | | | |
| Wall-clock duration | | | | |

🔮 **Predict before you look,** then fill the table with your own numbers. Which rows change between run 1 and run 2, and which do not?

**How to read it**

- **Files and bytes are the cause.** They are the same in run 1 and run 2: pruning is decided from the log, not from any cache.
- **Task time is the cost.** It is the sum of the time of every task. Compare it **run 1 with run 1** and **run 2 with run 2**: only then is the disk cache equal on both sides.
- **Rows read are (almost) the same in A and B.** Photon applies the date filter inside the Scan, so both Scans hand over only the rows of that week. Rows do not tell the two layouts apart; **files and bytes do**. What A pays for is opening and decoding every file to find those rows.
- **Two clocks.** Wall-clock is the time you wait: it also includes planning, reading the log and scheduling, and it depends on how many tasks the platform runs in parallel. On a heavily pruned query (B reads one file, so one task) task time can be **lower** than wall-clock: the fixed cost dominates. On a query that reads many files (A) task time is usually **higher** than wall-clock: many tasks ran at the same time. *Lab 2402* comes back to the two clocks.

### 9.6 Why: look inside the files

Now that the measurements are done (reading the data earlier would have warmed the cache), the same min/max picture as in Part A, on 200 M rows:

In [ ]:
%python
show_files(ORDERS_UNORDERED, "order_date")

In [ ]:
%python
show_files(ORDERS_CLUSTERED, "order_date")

Every file of A spans **2023-01-01 → 2024-12-31**: for any week, the log says "maybe here" for all of them. Every file of B spans about a month: for one week, the log points at one file (two at most, if the week crosses a boundary).

## Takeaway

> Data skipping only works if similar values sit together in the same files.
> **OPTIMIZE** fixes the size of the files, not their order.
> **Z-order** sorts them, but every load re-sorts the whole table (or partition).
> **Partitioning** sorts with folders, for one low-cardinality column.
> **Liquid clustering** sorts inside the files, for the columns you filter on, and keeps it up to date **incrementally**. It is the **recommended** layout for new tables.

## 🛠 Apply it to your tables

**Is my table full of small files?**
```sql
DESCRIBE DETAIL my_catalog.my_schema.my_table;
-- sizeInBytes / numFiles: a few MB per file → OPTIMIZE; hundreds of MB → healthy
```

**Creating a new table?** Cluster by the columns you filter on, not by partitions:
```sql
CREATE TABLE my_catalog.my_schema.my_table (...) CLUSTER BY (plant_id, event_date);
-- or CLUSTER BY AUTO: Unity Catalog managed tables with Predictive Optimization pick the keys for you
```

**Migrating a partitioned table?** It is a rewrite, not a setting:
```sql
CREATE TABLE my_catalog.my_schema.my_table_v2 CLUSTER BY (event_date)
AS SELECT * FROM my_catalog.my_schema.my_table;
```

**Changed the clustering key, or did a big backfill?** `OPTIMIZE my_table FULL` (Databricks Runtime 16.0+).

**Does my favourite filter use the layout?** Run it and open the Query Profile: if *files pruned* is close to 0 on a large table, cluster by that column.

**Who runs OPTIMIZE?** `DESCRIBE TABLE EXTENDED` → is Predictive Optimization enabled? If not, schedule `OPTIMIZE`.